# yolo26_cpp — train with **cuDNN** (GPU) → infer → show the image
End-to-end on one T4: fine-tune yolo26n on COCO128 using the **device engine with the cuDNN conv
backend** (`-DUSE_CUDA -DUSE_CUDNN`), save `best.pt`, then run CPU inference (`detect26`, NMS-free
one2one head) on `assets/bus.jpg` — for **both** the pretrained and the cuDNN-fine-tuned weights —
and show them side by side.
**Runtime → GPU (T4)**, then Run all.


In [ ]:
!nvidia-smi -L
!nvcc --version | tail -1


In [ ]:
%cd /content
!rm -rf yolo26_cpp
!git clone -q https://github.com/yomei-o/yolo26_cpp.git
%cd /content/yolo26_cpp
!pip -q install ultralytics


### Refs + pretrained init weights


In [ ]:
!python -c "from ultralytics import YOLO; YOLO('yolo26n.pt')"   # download pretrained yolo26n.pt
!python pure/ref/export_yolo26.py 64 yolo26n         # fused refs (uses pretrained)
!python pure/ref/export_unfused26.py 64 yolo26n      # unfused manifest/names + arch26 (pretrained bins)
!g++ -O2 -std=c++17 -Ipure/third_party pure/make_init_pt.cpp -o make_init_pt
!./make_init_pt init26_pre.pt from yolo26n.pt pure/ref/data_net/   # pretrained device init


### Locate cuDNN (auto)


In [ ]:
import os, glob
inc=lib=None
try:
    import nvidia.cudnn; d=os.path.dirname(nvidia.cudnn.__file__)
    if os.path.exists(d+"/include/cudnn.h"): inc,lib=d+"/include",d+"/lib"
except Exception: pass
if not inc:
    for h in ["/usr/include/cudnn.h"]+glob.glob("/usr/include/**/cudnn.h",recursive=True)+glob.glob("/usr/local/cuda*/include/cudnn.h"):
        if os.path.exists(h): inc,lib=os.path.dirname(h),"/usr/lib/x86_64-linux-gnu"; break
os.environ["CUDNN_INC"],os.environ["CUDNN_LIB"]=inc or "",lib or ""
print("CUDNN_INC =",inc,"\nCUDNN_LIB =",lib)


### 1. Train with cuDNN (`-DUSE_CUDA -DUSE_CUDNN -DUSE_CUBLAS`)
COCO128 from pretrained, imgsz 320, **lr 1e-4** (gentle — a from-scratch lr of 1e-3 drifts the pretrained model's confidence calibration and it stops detecting `bus.jpg`; 1e-4 keeps it sharp while the loss still ticks down). Saves `best.pt`/`last.pt`.


In [ ]:
!wget -q https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip && unzip -q -o coco128.zip
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA -DUSE_CUDNN -DUSE_CUBLAS -diag-suppress 550 \
      -I"$CUDNN_INC" -L"$CUDNN_LIB" -Ipure/third_party pure/dtrain_coco26.cpp -lcudnn -lcublas -o dtrain26_cudnn
!LD_LIBRARY_PATH="$CUDNN_LIB:$LD_LIBRARY_PATH" ./dtrain26_cudnn coco128/images/train2017 320 8 8 init26_pre.pt pure/ref/data_net/ 1e-4


### 2. Inference — pretrained vs cuDNN-fine-tuned (`detect26`, CPU host, NMS-free o2o)
`best.pt` is the unfused state-dict the device trainer saved — the CPU inferencer loads it directly (same 594-tensor manifest).


In [ ]:
!g++ -O2 -std=c++17 -Ipure/third_party pure/detect26.cpp -o detect26
print("===== pretrained (before training) =====")
!./detect26 init26_pre.pt assets/bus.jpg 640 0.25 det_pre.png pure/ref/data_net/
print("\n===== cuDNN fine-tuned (after training) =====")
!./detect26 best.pt        assets/bus.jpg 640 0.25 det_ft.png  pure/ref/data_net/


### 3. Show both results


In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg
fig,ax=plt.subplots(1,2,figsize=(13,9))
for a,(f,t) in zip(ax,[("det_pre.png","pretrained yolo26n"),("det_ft.png","cuDNN fine-tuned (COCO128)")]):
    a.imshow(mpimg.imread(f)); a.axis("off"); a.set_title(t)
plt.tight_layout(); plt.show()


Both should box the bus + several people. This closes the loop: **GPU/cuDNN training → inference → image** in pure C++ with no Python at run time.

**If the fine-tuned side shows fewer/no boxes:** the demo lr (1e-4) is already gentle, but 8 epochs on just 128 images is a strong fine-tune — lower `conf` (e.g. `0.1`) in the detect call, or drop lr to `1e-5` / fewer epochs. The pretrained side is unaffected and proves the inference path.
